# Fine tuning

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
!pip install unsloth
import unsloth
from unsloth import FastModel
import gc
from tqdm import tqdm
import re
import os
from google.colab import drive

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 157.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
import gc
from tqdm import tqdm
import re
import os
from google.colab import drive
import json

In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

RECORES_DATASET_PATH = os.path.join(BASE_PATH, "dataset-recores/")
QUESTIONS_FILE = os.path.join(BASE_PATH, "data/multiple_choice.json")
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_images.json")


Mounted at /content/drive


In [ ]:
#@title Carga del dataset ReCoRES y creación de los conjuntos de entrenamiento, validación y test
import pandas as pd

train_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'train.csv'), sep='\t')
val_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'dev.csv'), sep='\t')
test_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'test.csv'), sep='\t')

print("--- Información del Conjunto de Entrenamiento ---")
train_df.info()


--- Información del Conjunto de Entrenamiento ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1047 entries, 0 to 1046
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      1047 non-null   object
 1   question  1047 non-null   object
 2   A         1047 non-null   object
 3   B         1047 non-null   object
 4   C         1047 non-null   object
 5   D         1047 non-null   object
 6   E         1047 non-null   object
 7   answer    1047 non-null   object
 8   reason    1047 non-null   object
dtypes: object(9)
memory usage: 73.7+ KB


In [ ]:

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-9B",
    dtype = None,
    max_seq_length = 2048,
    load_in_4bit = False,
    full_finetuning = False,
)

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    bias = "none",
    random_state = 3407,
)

In [ ]:
import json
from datasets import Dataset
SYSTEM_PROMPT_MULTIMODAL = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.

Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

def build_conversation(row):
    user_content = f"Texto: {row['text']}\nPregunta: {row['question']}\nOpciones:\n"
    user_content += f"A) {row['A']}\nB) {row['B']}\nC) {row['C']}\nD) {row['D']}\nE) {row['E']}"

    assistant_content = str(row['answer']).strip()

    return [
        {"role": "system", "content": SYSTEM_PROMPT_MULTIMODAL},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content}
    ]

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts }

from datasets import Dataset

def prepare_dataset(df_input):
    """Función para automatizar la limpieza y formateo de cualquier split"""
    df_temp = df_input.copy()
    df_temp["conversations"] = df_temp.apply(build_conversation, axis=1)
    hf_ds = Dataset.from_pandas(df_temp[["conversations"]])

    return hf_ds.map(formatting_prompts_func, batched=True)

train_dataset = prepare_dataset(train_df)
val_dataset   = prepare_dataset(val_df)
test_dataset  = prepare_dataset(test_df)

Map:   0%|          | 0/1047 [00:00<?, ? examples/s]

Map:   0%|          | 0/363 [00:00<?, ? examples/s]

Map:   0%|          | 0/386 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset[0]["text"])

<|im_start|>system
Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.

Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra.<|im_end|>
<|im_start|>user
Texto: El trabajo es en primer término un proceso entre la naturaleza y el hombre, proceso en que este realiza, regula y controla mediante su propia acción su intercambio de materias con la naturaleza. En este proceso, el hombre se enfrenta como un poder natural con la materia de la naturaleza. Pone en acción las fuerzas naturales que forman su corporeidad, los brazos y las piernas

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",

        per_device_train_batch_size = 8,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 2,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,

        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_qwen35_final",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,047 | Num Epochs = 3 | Total steps = 198
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 86,556,672 of 9,496,370,416 (0.91% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.087036,0.054719
2,0.143335,0.060127
3,0.000495,0.062614


In [ ]:
lora_path = os.path.join(BASE_PATH, "qwen35_lora_solo_letra")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"Adaptadores guardados en: {lora_path}")

Adaptadores guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_lora_solo_letra


## Probar si funciona

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

RECORES_DATASET_PATH = os.path.join(BASE_PATH, "dataset-recores/")
QUESTIONS_FILE = os.path.join(BASE_PATH, "data/multiple_choice.json")
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_100.json")


In [ ]:
from unsloth import FastVisionModel

In [ ]:
model_path = os.path.join(BASE_PATH, "model_lora_final")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_lora_solo_letra",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
)

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
FastVisionModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1152)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-26): 27 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear4bit(in_features=1152, out_features=3456, bias=True)
                (proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear4bit(in_features=1152, out_features=4304, bias=True)
   

In [ ]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open(QUESTIONS_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open(ANSWERS_FILE, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    return data, ground_truth

In [ ]:
def filter_questions(data, ground_truth):
    """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']

            for q in exercise['questions']:
                q_id = q['questionId']

                if q_id in ground_truth:
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": q['options'], # Pasamos la lista original intacta
                        "real": ground_truth[q_id]
                    })
    return tareas

In [ ]:
from PIL import Image

def prepare_batch(batch, system_prompt, modo_salida):
    """
    Construye los mensajes en formato multimodal para un lote de tareas.
    Devuelve la lista de mensajes y una lista paralela con las imágenes cargadas.
    """
    mensajes_batch = []
    imagenes_batch = []

    for t in batch:
        user_content = []
        imagenes_tarea = []

        base_text = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n"
        user_content.append({"type": "text", "text": base_text})

        for opt in t['opciones']:
            letra = opt['optionId']
            texto = opt.get('text', '').strip()
            ruta_img = opt.get('image-path', '')

            if texto:
                user_content.append({"type": "text", "text": f"{letra}) {texto}\n"})

            elif ruta_img:
                user_content.append({"type": "text", "text": f"{letra}) "})


                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)


                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        if modo_salida == "letra":
            user_content.append({"type": "text", "text": "\nRespuesta:"})

        mensajes_batch.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [ ]:
import torch

def generate_response(model, processor, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    textos_prompt = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]

    # ---------------------------------------------------------
    # THE FIX: Access the inner .tokenizer attribute!
    # ---------------------------------------------------------
    if hasattr(processor, "tokenizer"):
        processor.tokenizer.padding_side = "left"
    else:
        processor.padding_side = "left" # Fallback just in case

    model_inputs = processor(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            # Ensure you are also targeting the inner tokenizer's pad token
            pad_token_id=processor.tokenizer.pad_token_id if hasattr(processor, "tokenizer") else processor.pad_token_id
        )
    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []

    for output in outputs:
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [ ]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación según el formato esperado."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    if modo_salida == "json":
        explicacion = texto_bruto
        try:
            json_match = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
            if json_match:
                datos = json.loads(json_match.group(0))
                letra_raw = datos.get("respuesta", "").strip().upper()
                match_letra = re.search(r'[A-D]', letra_raw)
                prediccion = match_letra.group(0) if match_letra else "N/A"
                explicacion = datos.get("razonamiento", "")
            else:
                error_formato = True
        except Exception:
            error_formato = True

    elif modo_salida == "letra":
        texto_bruto = texto_bruto.upper()
        match = re.search(r'[A-D]', texto_bruto)
        prediccion = match.group(0) if match else "N/A"
        if not match:
            error_formato = True
    return prediccion, explicacion, error_formato

In [ ]:
def show_results(stats, output_file):
    """Imprime por pantalla el resumen de la evaluación."""
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*50)
    print(f"RESULTADOS : {output_file}")
    print("="*50)
    if stats["errores_formato"] > 0:
        print(f"Errores de formato (JSON/Regex fallido): {stats['errores_formato']} de {stats['total']}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 50)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")

In [ ]:
def evaluate_model(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file="resultados.jsonl"
):
    """Función principal que orquesta todo el flujo."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    tareas = filter_questions(data, ground_truth)

    resultados_finales = []
    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    if os.path.exists(output_file):
        os.remove(output_file)

    for i in tqdm(range(0, len(tareas), batch_size), desc="Progreso"):
        batch = tareas[i : i + batch_size]

        mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida)
        textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

        batch_results = []

        for j, texto_bruto in enumerate(textos_generados):
            prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

            tarea_actual = batch[j]
            real = tarea_actual["real"]
            nivel = tarea_actual["nivel"]
            es_correcto = (prediccion == real)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1
            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1


            if errors:
                stats["errores_formato"] += 1

            batch_results.append({
                "questionId": tarea_actual["id"],
                "nivel": nivel,
                "pregunta": tarea_actual["pregunta"],
                "respuesta_real": real,
                "prediccion_modelo": prediccion,
                "explicacion": explicacion,
                "error_procesamiento_json": errors,
                "estado": "CORRECTO" if es_correcto else "INCORRECTO"
            })

        with open(output_file, 'a', encoding='utf-8') as f:
          for resultado in batch_results:
              linea_json = json.dumps(resultado, ensure_ascii=False)
              f.write(linea_json + '\n')

    show_results(stats, output_file)

In [ ]:
def split_tasks_by_modality(tareas):
    """Separa las tareas en dos grupos: de solo texto y con imágenes."""
    tareas_texto = []
    tareas_imagen = []

    for t in tareas:
        tiene_imagen = False
        if isinstance(t.get('opciones'), list):
            tiene_imagen = any(opt.get('image-path', '') != '' for opt in t['opciones'])

        if tiene_imagen:
            tareas_imagen.append(t)
        else:
            tareas_texto.append(t)

    return tareas_texto, tareas_imagen


def run_inference(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size_texto=4,
    output_file="resultados.jsonl"
):
    """Ejecuta la inferencia procesando primero texto en batches y luego imágenes 1 a 1."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    todas_las_tareas = filter_questions(data, ground_truth)
    tareas_texto, tareas_imagen = split_tasks_by_modality(todas_las_tareas)

    if os.path.exists(output_file):
        os.remove(output_file)

    def procesar_grupo(grupo_tareas, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_tareas), b_size), desc=descripcion):
            batch = grupo_tareas[i : i + b_size]

            mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida)
            textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

                tarea_actual = batch[j]
                real = tarea_actual["real"]
                nivel = tarea_actual["nivel"]
                es_correcto = (prediccion == real)

                batch_results.append({
                    "questionId": tarea_actual["id"],
                    "nivel": nivel,
                    "pregunta": tarea_actual["pregunta"],
                    "respuesta_real": real,
                    "prediccion_modelo": prediccion,
                    "respuesta_completa": texto_bruto,
                    "explicacion": explicacion,
                    "error_procesamiento_json": errors,
                    "estado": "CORRECTO" if es_correcto else "INCORRECTO"
                })

            with open(output_file, 'a', encoding='utf-8') as f:
                for resultado in batch_results:
                    linea_json = json.dumps(resultado, ensure_ascii=False)
                    f.write(linea_json + '\n')

    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} tareas de SOLO TEXTO (Batch Size: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto")

    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} tareas MULTIMODALES (Batch Size: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes")

    print(f"\nResultados guardados en: {output_file}")

In [ ]:
def calculate_metrics(input_file="resultados.jsonl"):
    """Lee las predicciones almacenadas y calcula las métricas finales."""

    if not os.path.exists(input_file):
        print(f"Error: No se ha encontrado el archivo {input_file}.")
        return

    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    with open(input_file, 'r', encoding='utf-8') as f:
        for linea in f:
            if not linea.strip():
                continue

            # Cargar el JSON de la línea actual
            resultado = json.loads(linea)

            nivel = resultado["nivel"]
            es_correcto = (resultado["estado"] == "CORRECTO")
            error_json = resultado.get("error_procesamiento_json", False)

            # Inicializar el nivel si no existe en las estadísticas
            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            # Contabilizar globales y por nivel
            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1

            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1

            if error_json:
                stats["errores_formato"] += 1

    # Llamar a tu función original para mostrar los resultados en pantalla/guardarlos
    show_results(stats, input_file)

In [ ]:
def filter_and_calculate(input_file='prueba.json'):
    with open(ANSWERS_FILE, 'r', encoding='utf-8') as f:
        subset_data = json.load(f)
        allowed_ids = set(subset_data.keys())

    output_path = 'filtered_results.jsonl'

    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:

        for linea in f_in:
            if not linea.strip():
                continue
            try:
                entry = json.loads(linea)
                if entry.get("questionId") in allowed_ids:
                    f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
            except json.JSONDecodeError:
                continue

    calculate_metrics(output_path)

In [ ]:
qwen35_path = os.path.join(BASE_PATH, 'qwen35_results')

In [ ]:
qwen35_results_path = os.path.join(qwen35_path, "qwen35_zero_shot_fine_tuning.json")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT_MULTIMODAL = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.

Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.
No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Solo la letra."""

run_inference(
    model=model,
    tokenizer=tokenizer,
    system_prompt=SYSTEM_PROMPT_MULTIMODAL,
    modo_salida="letra",
    batch_size_texto=4,
    output_file=qwen35_results_path
)

calculate_metrics(input_file=qwen35_results_path)


--- Procesando 128 tareas de SOLO TEXTO (Batch Size: 4) ---


Progreso Texto: 100%|██████████| 32/32 [01:13<00:00,  2.29s/it]



--- Procesando 7 tareas MULTIMODALES (Batch Size: 1) ---


Progreso Imágenes: 100%|██████████| 7/7 [01:02<00:00,  8.92s/it]


Resultados guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_results/qwen35_zero_shot_fine_tuning.json

RESULTADOS : /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/qwen35_results/qwen35_zero_shot_fine_tuning.json
Accuracy Global: 86.67% (117/135)
--------------------------------------------------
Nivel A1: 90.14% (64/71)
Nivel A2: 82.61% (19/23)
Nivel B1: 80.95% (17/21)
Nivel B2: 85.00% (17/20)


In [ ]:
#@title Accuracy solo sobre ejemplos con imágenes
filter_and_calculate(input_file=qwen35_results_path)


RESULTADOS : filtered_results.jsonl
Accuracy Global: 100.00% (7/7)
--------------------------------------------------
Nivel A1: 100.00% (7/7)
